# Intent Classification Model

This notebook trains and evaluates an intent classification model using **prototype matching** with **heuristic labels** for evaluation.

## Model Architecture
- **Feature Extraction**: TF-IDF Vectorizer
- **Classifier**: Prototype-based matching (cosine similarity)
- **Task**: 4-class classification (inform, persuade, entertain, deceive)

## Evaluation Method
Uses **heuristic labels** (rule-based) to approximate accuracy metrics.

In [3]:
# Imports
import pandas as pd
import numpy as np
import re, csv, warnings, joblib
from collections import Counter
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

print('✓ All imports loaded!')

✓ All imports loaded!


## 1. Data Loading

In [4]:
COLS = ['id','label','statement','subjects','speaker','job_title',
        'state_info','party_affiliation','barely_true_cnt','false_cnt',
        'half_true_cnt','mostly_true_cnt','pants_on_fire_cnt','context','justification']

def read_tsv(path):
    return pd.read_csv(path, sep='\t', header=None, names=COLS, engine='python',
                       quoting=csv.QUOTE_NONE, escapechar='\\', on_bad_lines='skip')

def text_of(r):
    return ' '.join([str(r.get('statement','')), str(r.get('context','')), str(r.get('justification',''))]).strip()

# Load datasets
df_tr = read_tsv('../data/train2.tsv')
df_va = read_tsv('../data/val2.tsv')
df_te = read_tsv('../data/test2.tsv')

# Prepare texts
for c in ['statement','context','justification']:
    df_tr[c] = df_tr[c].fillna('').astype(str).str.strip()
    df_va[c] = df_va[c].fillna('').astype(str).str.strip()
    df_te[c] = df_te[c].fillna('').astype(str).str.strip()

train_texts = df_tr.apply(text_of, axis=1).tolist()
val_texts = df_va.apply(text_of, axis=1).tolist()
test_texts = df_te.apply(text_of, axis=1).tolist()

print(f'Train: {len(train_texts)}, Val: {len(val_texts)}, Test: {len(test_texts)}')

Train: 10300, Val: 1284, Test: 1299


## 2. Model Training

In [5]:
# Train TF-IDF
intent_tfidf = TfidfVectorizer(lowercase=True, strip_accents='unicode', analyzer='word',
                                ngram_range=(1,2), min_df=3, max_df=0.95, sublinear_tf=True)
X_train = intent_tfidf.fit_transform(train_texts)
print(f'✓ Vocabulary: {len(intent_tfidf.vocabulary_)}, Matrix: {X_train.shape}')

✓ Vocabulary: 62909, Matrix: (10300, 62909)


In [6]:
# Define prototypes
PROTOS = {
    'inform':   ['Officials said the department released a report with data.',
                 'The study shows the unemployment rate decreased.',
                 'According to the census, population grew by 2.5 percent.'],
    'persuade': ['We should support this policy because it will improve outcomes.',
                 'This legislation is crucial for protecting our environment.',
                 'Vote yes to ensure better schools.'],
    'entertain': ['The comedian joked about daily life.',
                  'The talk show host made the audience laugh.',
                  'The satirical piece poked fun at absurdities.'],
    'deceive':  ['You won\'t believe this miracle cure doctors hate.',
                 'This weird trick will make you rich overnight.',
                 'Media doesn\'t want you to know this.']
}

CLASS_NAMES = ['inform','persuade','entertain','deceive']

# Create prototype matrix
proto_rows = []
for name in CLASS_NAMES:
    pv = intent_tfidf.transform(PROTOS[name])
    pv_mean = np.asarray(pv.mean(axis=0))
    pv_norm = normalize(pv_mean, norm='l2', axis=1)
    proto_rows.append(pv_norm.ravel())

PROTO_MAT = np.vstack(proto_rows)
print(f'✓ Prototype matrix: {PROTO_MAT.shape}')

✓ Prototype matrix: (4, 62909)


## 3. Prediction Functions

In [7]:
def predict_intent(text):
    """Predict intent using prototype matching"""
    z = intent_tfidf.transform([text])
    zn = normalize(z, norm='l2', axis=1)
    scores = (zn @ PROTO_MAT.T).ravel()
    by_label = {CLASS_NAMES[i]: float(scores[i]) for i in range(4)}
    return max(by_label, key=by_label.get), by_label

def predict_batch(texts):
    """Predict intents for multiple texts"""
    return [predict_intent(t)[0] for t in texts]

print('✓ Prediction functions defined!')

✓ Prediction functions defined!


## 4. Create Heuristic Labels for Evaluation

**Note**: These are rule-based labels (not true human annotations).
They provide an approximate evaluation metric.

In [8]:
def create_heuristic_labels(df):
    """Create pseudo-labels based on keyword heuristics"""
    labels = []
    for _, row in df.iterrows():
        stmt = str(row.get('statement', '')).lower()
        context = str(row.get('context', '')).lower()
        combined = stmt + ' ' + context
        
        # Rule-based classification
        if any(w in combined for w in ['should', 'must', 'need to', 'have to', 'vote', 'support', 'crucial', 'important']):
            labels.append('persuade')
        elif any(w in combined for w in ['miracle', 'secret', 'shocking', 'unbelievable', 'trick', 'hate']):
            labels.append('deceive')
        elif any(w in combined for w in ['joke', 'funny', 'laugh', 'humor', 'comedy', 'hilarious']):
            labels.append('entertain')
        else:
            labels.append('inform')
    return labels

# Create heuristic labels
y_train_h = create_heuristic_labels(df_tr)
y_val_h = create_heuristic_labels(df_va)
y_test_h = create_heuristic_labels(df_te)

print('✓ Heuristic labels created')
print('\nHeuristic Label Distribution (Train):')
print(Counter(y_train_h))

✓ Heuristic labels created

Heuristic Label Distribution (Train):
Counter({'inform': 8924, 'persuade': 1246, 'deceive': 115, 'entertain': 15})


## 5. Model Evaluation

In [9]:
# Make predictions
print('Making predictions...')
y_pred_train = predict_batch(train_texts)
y_pred_val = predict_batch(val_texts)
y_pred_test = predict_batch(test_texts)

print('\nPrediction Distribution:')
print('Train:', Counter(y_pred_train))
print('Val:  ', Counter(y_pred_val))
print('Test: ', Counter(y_pred_test))

Making predictions...

Prediction Distribution:
Train: Counter({'persuade': 3915, 'inform': 3415, 'deceive': 1750, 'entertain': 1220})
Val:   Counter({'persuade': 484, 'inform': 401, 'deceive': 238, 'entertain': 161})
Test:  Counter({'persuade': 496, 'inform': 427, 'deceive': 210, 'entertain': 166})


In [10]:
# Calculate accuracy (vs heuristic labels)
print('\n' + '='*60)
print('ACCURACY SCORES (vs. Heuristic Labels)')
print('='*60)
print('Note: These scores are relative to rule-based labels, not true labels.')
print()

acc_train = accuracy_score(y_train_h, y_pred_train)
acc_val = accuracy_score(y_val_h, y_pred_val)
acc_test = accuracy_score(y_test_h, y_pred_test)

print(f'Training Accuracy:   {acc_train:.4f} ({acc_train*100:.2f}%)')
print(f'Validation Accuracy: {acc_val:.4f} ({acc_val*100:.2f}%)')
print(f'Test Accuracy:       {acc_test:.4f} ({acc_test*100:.2f}%)')


ACCURACY SCORES (vs. Heuristic Labels)
Note: These scores are relative to rule-based labels, not true labels.

Training Accuracy:   0.3565 (35.65%)
Validation Accuracy: 0.3660 (36.60%)
Test Accuracy:       0.3603 (36.03%)


In [11]:
# Macro-averaged metrics
print('\n' + '='*60)
print('MACRO-AVERAGED METRICS')
print('='*60)

for split_name, y_true, y_pred in [('Training', y_train_h, y_pred_train),
                                     ('Validation', y_val_h, y_pred_val),
                                     ('Test', y_test_h, y_pred_test)]:
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    print(f'\n{split_name}:')
    print(f'  Precision: {p:.4f}')
    print(f'  Recall:    {r:.4f}')
    print(f'  F1-Score:  {f1:.4f}')


MACRO-AVERAGED METRICS

Training:
  Precision: 0.2662
  Recall:    0.3961
  F1-Score:  0.1912

Validation:
  Precision: 0.2855
  Recall:    0.4344
  F1-Score:  0.2043

Test:
  Precision: 0.2697
  Recall:    0.3578
  F1-Score:  0.1950


In [12]:
# Classification report (Validation)
print('\n' + '='*60)
print('CLASSIFICATION REPORT (Validation Set)')
print('='*60)
print(classification_report(y_val_h, y_pred_val, zero_division=0))


CLASSIFICATION REPORT (Validation Set)
              precision    recall  f1-score   support

     deceive       0.01      0.27      0.02        11
   entertain       0.01      0.50      0.01         2
      inform       0.94      0.33      0.49      1127
    persuade       0.19      0.63      0.29       144

    accuracy                           0.37      1284
   macro avg       0.29      0.43      0.20      1284
weighted avg       0.84      0.37      0.46      1284



In [13]:
# Confusion matrix
print('\n' + '='*60)
print('CONFUSION MATRIX (Validation Set)')
print('='*60)
cm = confusion_matrix(y_val_h, y_pred_val, labels=CLASS_NAMES)
cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
print('\nRows = True Labels (heuristic), Columns = Predictions')
print(cm_df)


CONFUSION MATRIX (Validation Set)

Rows = True Labels (heuristic), Columns = Predictions
           inform  persuade  entertain  deceive
inform        375       390        142      220
persuade       22        91         16       15
entertain       1         0          1        0
deceive         3         3          2        3
